In [ ]:
from pathlib import Path
import pandas as pd

# =========================
# 1. File paths
# =========================

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent

BOYCOTT_RAW_PATH = PROJECT_ROOT / "data" / "raw" / "x_2022_boycott_raw_master.csv"
CANCEL_RAW_PATH = PROJECT_ROOT / "data" / "raw" / "x_2022_cancel_raw_master.csv"

REPORTS_DIR = PROJECT_ROOT / "data" / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

BOYCOTT_OUTPUT_PATH = REPORTS_DIR / "x_2022_boycott_collected_monthly_counts.csv"
CANCEL_OUTPUT_PATH = REPORTS_DIR / "x_2022_cancel_collected_monthly_counts.csv"


# =========================
# 2. Function
# =========================

def make_monthly_count_csv(raw_path, hashtag_type, output_path):
    """
    Create monthly collected count CSV from raw master file.
    Counts both total rows and unique post_id rows.
    """
    
    df = pd.read_csv(raw_path, dtype="string", keep_default_na=False)
    
    # Parse timestamp
    df["created_at"] = pd.to_datetime(df["created_at"], utc=True, errors="coerce")
    
    # Keep only 2022 rows
    df = df[
        (df["created_at"] >= pd.Timestamp("2022-01-01T00:00:00Z")) &
        (df["created_at"] < pd.Timestamp("2023-01-01T00:00:00Z"))
    ].copy()
    
    # Create month column
    df["month"] = df["created_at"].dt.to_period("M").astype(str)
    
    # Monthly count
    monthly = (
        df.groupby("month")
        .agg(
            collected_rows=("post_id", "count"),
            unique_post_count=("post_id", "nunique")
        )
        .reset_index()
    )
    
    monthly.insert(0, "hashtag_type", hashtag_type)
    
    # Make sure all 12 months appear
    all_months = pd.DataFrame({
        "month": pd.period_range("2022-01", "2022-12", freq="M").astype(str)
    })
    
    monthly = all_months.merge(monthly, on="month", how="left")
    monthly["hashtag_type"] = monthly["hashtag_type"].fillna(hashtag_type)
    monthly["collected_rows"] = monthly["collected_rows"].fillna(0).astype(int)
    monthly["unique_post_count"] = monthly["unique_post_count"].fillna(0).astype(int)
    
    monthly = monthly[
        ["hashtag_type", "month", "collected_rows", "unique_post_count"]
    ]
    
    # Save CSV
    monthly.to_csv(output_path, index=False, encoding="utf-8-sig")
    
    print(f"Saved: {output_path}")
    print(f"Total collected rows: {monthly['collected_rows'].sum():,}")
    print(f"Total unique posts: {monthly['unique_post_count'].sum():,}")
    
    return monthly


# =========================
# 3. Run
# =========================

boycott_monthly = make_monthly_count_csv(
    BOYCOTT_RAW_PATH,
    "boycott",
    BOYCOTT_OUTPUT_PATH
)

cancel_monthly = make_monthly_count_csv(
    CANCEL_RAW_PATH,
    "cancel",
    CANCEL_OUTPUT_PATH
)